In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import time
from functools import partial
import qutip.settings
qutip.settings.num_threads = 4
#import sympy as sp
import multiprocessing as mp
import functools
from scipy.sparse import csc_matrix
import scipy.sparse.linalg as sla
import os
from tqdm import tqdm

def DH(w, w0, g, M, j):
    
    '''
    This function returns the Dicke Hamiltonian for the following parameters.
    
    Parameters
    ----------
    
    w : frequency of the bosonic field
    
    w0 : Energy difference in spin states
    
    g : Coupling strength
    
    M : Upper limit of bosonic fock states
    
    j : Pseudospin
    
    '''
    a  = qt.tensor(qt.destroy(M),qt.qeye(int(2*j+1)))
    Jp = qt.tensor(qt.qeye(M),qt.jmat(j, '+'))
    Jm = qt.tensor(qt.qeye(M),qt.jmat(j, '-'))
    Jz = qt.tensor(qt.qeye(M),qt.jmat(j, 'z'))
    
    H0 = w * a.dag() * a + w0 * Jz
    H1 = 2.0 / np.sqrt(j) * (a + a.dag()) * (Jp + Jm)
    H = H0 + g * H1
    
    return H

def dSFF_fun(j,M,g,beta,eval_list,tlist,slist):

    if not os.path.exists("SFF"):
        os.mkdir("SFF")

    with open(f"SFF/dSFFvsTime,j={j},M={M},g={g},beta={beta}.dat", 'w') as file:
        for t_ind, t in tqdm(enumerate(tlist)):
            for s_ind, s in enumerate(slist):
                SFF = 0
                norm = 0
                for i,eval1 in (enumerate(eval_list)):
                    SFF += np.exp(-(beta+1j*t)*(np.real(eval1))-(beta+1j*s)*(np.imag(eval1)))
                    norm += np.exp(-beta*eval1)
                # SFF1 = SFF/norm
                SFF = np.conjugate(SFF)*SFF/(norm**2)
                file.write("\t{}".format(SFF))
                # file.write("\t{}".format(SFF1))
            file.write("\n")

    return 0

In [2]:
S = 10; wc = 1.0;ws=1.0;κ=2.0;
λ1 = 0.2;λ2 = 1.0;λc = 1/np.sqrt(2);
Nphot1 = 10
a  = qt.tensor(qt.destroy(Nphot1),qt.qeye(int(2*S+1)))
HDicke1_subrad = DH(wc, ws, λ1, Nphot1, S)
HDicke1_suprad = DH(wc, ws, λ2, Nphot1, S)
Lop1_subrad = qt.liouvillian(HDicke1_subrad,c_ops=[np.sqrt(κ)*a])
Lop1_suprad = qt.liouvillian(HDicke1_subrad,c_ops=[np.sqrt(κ)*a])
num  = qt.tensor(qt.num(Nphot1),qt.qeye(int(2*S+1)))
Sz = qt.tensor(qt.qeye(Nphot1),qt.jmat(S, 'z'))
β = 0

t_vals_0_to_01 = np.linspace(0, 0.1, 1000, endpoint=False)
t_vals_01_to_1 = np.linspace(0.1, 1, 1000, endpoint=False)
t_vals_1_to_10 = np.linspace(1, 10, 1000, endpoint=False)
t_vals_10_to_100 = np.linspace(10, 100, 1000, endpoint=False)
t_vals_100_to_1000 = np.linspace(100, 1000, 1000)

# Concatenate them into a single array
tlist = np.concatenate([t_vals_0_to_01, t_vals_01_to_1, t_vals_1_to_10, t_vals_10_to_100, t_vals_100_to_1000])

slist = tlist

In [3]:
Lopmatsubrad = Lop1_subrad.data
Lopmatsubrad = Lopmatsubrad.to_array()
Lopmatsubrad_even = Lopmatsubrad[::2,::2]
Lopmatsubrad_odd = Lopmatsubrad[1::2,1::2]

In [3]:
Lopmatsuprad = Lop1_suprad.data
Lopmatsuprad = Lopmatsuprad.to_array()
Lopmatsuprad_even = Lopmatsuprad[::2,::2]
Lopmatsuprad_odd = Lopmatsuprad[1::2,1::2]

In [ ]:
Ntot = (2*S+1)*Nphot1
Lopmatsubrad_even_eigs_sparse = sla.eigs(csc_matrix(Lopmatsubrad_even), k=Ntot-2, sigma=0+0j, which='LM', return_eigenvectors=False, maxiter=2000)
idx = np.argsort(abs(Lopmatsubrad_even_eigs_sparse.imag))
Lopmatsubrad_even_eigs_sparse = Lopmatsubrad_even_eigs_sparse[idx]
if not os.path.exists("evals_par_Lop"):
    os.mkdir("evals_par_Lop")
file = 'evals_par_Lop/evals_Lop_g={λ1}_j={j}_M={M}'
np.save(file,Lopmatsubrad_even_eigs_sparse)
eval_list = np.load(file)
dSFF_fun(S,Nphot1,λ1,β,eval_list,tlist,slist)

In [12]:
Ntot = (2*S+1)*Nphot1
Lopmatsuprad_even_eigs_sparse = sla.eigs(csc_matrix(Lopmatsuprad_even), k=Ntot-2, sigma=0+0j, which='LM', return_eigenvectors=False, maxiter=2000)
idx = np.argsort(abs(Lopmatsuprad_even_eigs_sparse.imag))
Lopmatsuprad_even_eigs_sparse = Lopmatsuprad_even_eigs_sparse[idx]
if not os.path.exists("evals_par_Lop"):
    os.mkdir("evals_par_Lop")
file = f'evals_par_Lop/evals_g={λ2}_j={S}_M={Nphot1}.npy'
np.save(file,Lopmatsuprad_even_eigs_sparse)
eval_list = np.load(file)
dSFF_fun(S,Nphot1,λ2,β,eval_list,tlist,slist)

0it [00:00, ?it/s]

5000it [8:04:26,  5.81s/it]


0